# B4 · Catching Mistakes

*Path B: AI-Assisted, notebook 4 of 5 · about 30 to 40 minutes*

**In this notebook you will learn to:**

- Get useful help from an AI assistant when code gives an error
- Spot code that runs without an error but gives a wrong answer
- Use a short set of checks that catch the most common mistakes

**Before this notebook:** [B3: Exploring and Plotting with AI](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/B3_Exploring_and_Plotting_with_AI.ipynb).

> **Before you start:** click **File > Save a copy in Drive** and work in your copy. Run a code cell by clicking the play button to its left, or press **Shift + Enter**. Go in order from top to bottom.

## 1. Two kinds of problem

Code can go wrong in two ways.

- **It stops with an error.** This is the easy kind. You know something is wrong, and the message tells you where to look.
- **It runs and gives a wrong answer.** This is the dangerous kind, because nothing tells you. AI-written code fails this way as readily as human-written code.

This notebook practices both.

Run the setup cell. The lines printed by `library(tidyverse)` and `read_csv()` are normal.

In [ ]:
library(tidyverse)

person_df    <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/demographic_data.csv")
condition_df <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/conditions.csv")
study_df     <- read_csv("https://raw.githubusercontent.com/icohr-lab/aou-workshop/refs/heads/main/data/final_dataset2.csv")

# ---- Helper code. You do not need to read this part; just run the cell. ----
describe_columns <- function(data) {
  tibble(column = names(data), type = map_chr(data, ~ class(.x)[1]))
}

## Part 1: When code gives an error

An AI assistant is very good at explaining errors, if you give it the right material.

**Give it:** the code that failed, the complete error message, and your column names and types from `describe_columns()`.

**Do not give it:** rows of data. With real data, that includes the output of `head()`, `str()`, or a printed dataframe. Read an error message before you paste it, and take out any data values it happens to quote.

A prompt that works:

> This R code gives an error. Explain what the error means and how to fix it.
>
> Code: *(paste the code)*
>
> Error: *(paste the whole message)*
>
> My dataframe's columns and types: *(paste the describe_columns() output)*

In [ ]:
describe_columns(study_df)

### Error 1

The cell below is **meant to fail**. Run it, then use the prompt above to find out why. Fix the code in the cell after it.

In [ ]:
study_df %>% fliter(age >= 65) %>% nrow()

In [ ]:
# Your corrected version

*Expected result after fixing: 34*

### Error 2

Also **meant to fail**. This time the error message is longer. The useful part is near the top.

In [ ]:
study_df %>%
  group_by(sex) %>%
  summarise(mean_SBP = mean(SBP, na.rm = TRUE))

In [ ]:
# Your corrected version

*Expected result after fixing: three rows, for Female, Male, and Other*

> **Also expect this:** AI assistants sometimes use a function that does not exist, or that only exists in a newer version of a package than the one installed. The symptom is `could not find function`. Tell the AI the exact error and ask for another way to do it.

## Part 2: When code runs but is wrong

Each example below shows a request and the code an AI assistant returned. **Every one runs without an error, and every one is wrong.** For each:

1. Run it and look at the result.
2. Use the clue to work out what is wrong.
3. Write a corrected version.

The Solutions section at the end explains each one. Try first.

### Mistake 1

**The request:** "What is the mean systolic blood pressure among men?"

In [ ]:
study_df %>%
  filter(sex_at_birth == "male") %>%
  summarise(n = n(), mean_SBP = mean(SBP, na.rm = TRUE))

(`n()` counts the rows, and `NaN` means "not a number".)

**Clue:** look at `n`. Then look at how the categories are spelled: `study_df %>% count(sex_at_birth)`.

In [ ]:
# Your corrected version

### Mistake 2

**The request:** "Simplify the race variable: change 'Black or African American' to 'Black' and 'Skip' to 'Unknown'."

In [ ]:
person_df %>% count(race)   # before

recoded <- person_df %>%
  mutate(race = case_when(
    race == "Black or African American" ~ "Black",
    race == "Skip" ~ "Unknown"
  ))

recoded %>% count(race)     # after

**Clue:** compare the two tables. What happened to everyone who was "Asian" or "White"?

In [ ]:
# Your corrected version

### Mistake 3

**The request:** "Add each person's diagnosis information to the demographics table."

`person_df` has 100 people.

In [ ]:
with_diagnosis <- person_df %>% left_join(condition_df, by = "person_id")
nrow(with_diagnosis)

**Clue:** compare the number of rows with `n_distinct(with_diagnosis$person_id)`. You met this problem in notebook F6.

In [ ]:
# Your corrected version

### Mistake 4

This one is the hardest to spot, because the result looks perfectly reasonable.

**The request:** "The mean of SBP comes back as NA. Fix it."

In [ ]:
study_df %>%
  mutate(SBP = replace_na(SBP, 0)) %>%
  summarise(mean_SBP = mean(SBP))

**Clue:** compare with `summary(study_df$SBP)`. Then ask yourself what `replace_na(SBP, 0)` does to the five people whose blood pressure was not measured.

In [ ]:
# Your corrected version

## Part 3: The checks that catch most mistakes

All four mistakes had something in common: the request left something out, the AI filled the gap with a guess, and a simple check would have caught it.

| After you... | Check | It catches |
|---|---|---|
| Filter rows | How many rows are left? | Filters that match nothing or everything |
| Recode a column | `count()` the column before and after | Lost categories, new missing values |
| Join tables | `nrow()` against `n_distinct(person_id)` | Dropped or duplicated people |
| Calculate a statistic | Compare with `summary()`; count the missing values | Missing values handled wrongly |
| Write rule-based logic | Try it on a few cases you know | Rules that are wrong or in the wrong order |

None of these checks takes more than a line of code.

---
## Recap

| Habit | Why |
|---|---|
| Give the AI the code, the full error, and the column names | It can explain and fix most errors |
| Never paste rows of real data | It violates the Data User Code of Conduct |
| Say how missing values should be handled | Otherwise the AI decides |
| Say "leave everything else unchanged" | Prevents side effects in recoding |
| Say what one row of each table represents | Prevents joins that duplicate people |
| Run a check after every step | Wrong code rarely announces itself |

---
## Solutions

Try each exercise before looking. There is often more than one correct way to write it.

**Error 1**

`fliter` is a typing mistake for `filter`. The message `could not find function` usually means a misspelled function name or a package that has not been loaded.

```r
study_df %>% filter(age >= 65) %>% nrow()
```

**Error 2**

There is no column called `sex`; it is called `sex_at_birth`. Giving the AI your `describe_columns()` output is what lets it spot this.

```r
study_df %>%
  group_by(sex_at_birth) %>%
  summarise(mean_SBP = mean(SBP, na.rm = TRUE))
```

**Mistake 1**

The category is `"Male"` with a capital M. `"male"` matched nobody, so the result is based on 0 people, and R gave no error. **The habit:** after a filter, check how many rows are left. **In the prompt:** give the AI the exact category values. Correct result: 48 men, mean SBP 118.5.

```r
study_df %>%
  filter(sex_at_birth == "Male") %>%
  summarise(n = n(), mean_SBP = mean(SBP, na.rm = TRUE))
```

**Mistake 2**

73 people now have `NA` for race. `case_when()` gives `NA` to every value that matches none of its rules, unless you add `.default = race` to keep the original value. **The habit:** count the categories before and after any recoding. **In the prompt:** say "leave all other values unchanged".

```r
recoded <- person_df %>%
  mutate(race = case_when(
    race == "Black or African American" ~ "Black",
    race == "Skip" ~ "Unknown",
    .default = race
  ))
recoded %>% count(race)
```

**Mistake 3**

166 rows for 100 people. `condition_df` has several records for some people, so the join repeats those people. Any count or average taken from this table would count them more than once. First reduce `condition_df` to one row per person, as you did in notebook B2. **The habit:** after every join, compare rows with different people. **In the prompt:** say what one row of each table represents.

```r
first_diagnosis <- condition_df %>%
  group_by(person_id) %>%
  slice_min(condition_start_date, n = 1, with_ties = FALSE) %>%
  ungroup()
with_diagnosis <- person_df %>% left_join(first_diagnosis, by = "person_id")
nrow(with_diagnosis)
```

**Mistake 4**

The code replaced the 5 missing values with 0 and then averaged all 100, giving 111.4. The correct mean of the 95 measured values is 117.2. A blood pressure of 0 is not "unknown"; it is a wrong number. The request was the real problem: "fix it" let the AI choose how. **The habit:** compare any statistic with `summary()`. **In the prompt:** say how missing values should be treated, for example "ignore missing values".

```r
study_df %>% summarise(mean_SBP = mean(SBP, na.rm = TRUE))
```

---
## Next

Continue with [B5: Statistics with AI](https://colab.research.google.com/github/icohr-lab/tutorials/blob/main/notebooks/B5_Statistics_with_AI.ipynb).